# FPL Hidden Gem Finder: Exploratory Analysis

This notebook documents the analysis behind the Streamlit dashboard. The goal is to find players who combine strong historical FPL output with a low price and low ownership.

The analysis uses the official FPL `bootstrap-static` API and the same hand-weighted score as the app. It intentionally does **not** train a machine-learning model.

## Analysis questions

1. How many players and positions are in the current API response?
2. Which players offer the most total points per £m?
3. Which affordable, low-owned players rank highest on the Gem Score?
4. Does the price-versus-points chart reveal useful budget picks?

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_cleaning import fetch_fpl_data
from src.feature_engineering import add_gem_score

pd.set_option("display.max_columns", 20)
players = fetch_fpl_data()
players.shape

In [ ]:
players[["web_name", "position_name", "now_cost", "total_points", "selected_by_percent"]].head(10)

## Data quality and position breakdown

Prices are converted from the API convention of tenths of a million (`55` becomes £5.5m). Numeric fields are converted explicitly before filtering or scoring.

In [ ]:
quality_summary = pd.DataFrame({
    "rows": [len(players)],
    "columns": [players.shape[1]],
    "missing_player_names": [players["web_name"].isna().sum()],
    "missing_prices": [players["now_cost"].isna().sum()],
    "missing_ownership": [players["selected_by_percent"].isna().sum()],
})
quality_summary

## Gem Score

The score combines two normalized signals:

- **75% points per million:** `total_points / now_cost`
- **25% inverse ownership:** `max_ownership - selected_by_percent`

Normalization puts both signals on a comparable 0–1 scale before weighting. This is a transparent ranking heuristic, not a prediction of future points.

In [ ]:
scored_players = add_gem_score(players)
scored_players["points_per_million"] = scored_players["points_per_million"].round(2)
scored_players["gem_score"] = scored_players["gem_score"].round(3)

scored_players[[
    "web_name",
    "position_name",
    "now_cost",
    "total_points",
    "selected_by_percent",
    "points_per_million",
    "gem_score",
]].sort_values("gem_score", ascending=False).head(10)

## Affordable, low-owned candidates

This example uses the same kind of shortlist a manager might explore in the app: players priced at or below £7m and owned by no more than 10% of managers.

In [ ]:
shortlist = scored_players[
    (scored_players["now_cost"] <= 7.0)
    & (scored_players["selected_by_percent"] <= 10.0)
].sort_values("gem_score", ascending=False)

shortlist[[
    "web_name",
    "position_name",
    "now_cost",
    "total_points",
    "selected_by_percent",
    "points_per_million",
    "gem_score",
]].head(15)

## Price versus total points

The chart below shows the main trade-off behind the project. Colour indicates position and labels appear for the affordable, low-owned shortlist.

In [ ]:
fig = px.scatter(
    scored_players,
    x="now_cost",
    y="total_points",
    color="position_name",
    hover_name="web_name",
    hover_data=["selected_by_percent", "gem_score"],
    labels={
        "now_cost": "Price (£m)",
        "total_points": "Total points",
        "position_name": "Position",
    },
    title="FPL players: price versus total points",
)
fig.update_traces(marker={"size": 8, "opacity": 0.7})
fig.show()

## Position-level summary

This summary helps check whether the score is concentrated in one position group.

In [ ]:
position_summary = (
    scored_players.groupby("position_name")
    .agg(
        players=("id", "count"),
        average_price=("now_cost", "mean"),
        average_points=("total_points", "mean"),
        average_gem_score=("gem_score", "mean"),
    )
    .round(2)
    .sort_values("average_gem_score", ascending=False)
)
position_summary

## Conclusions

The notebook provides a reproducible snapshot of the data behind the dashboard. The shortlist is a starting point for human FPL judgment: Gem Score rewards value and low ownership, but it does not account for fixtures, minutes risk, injuries, or future expected points. The Streamlit app exposes the price, ownership, and position filters interactively.

## Phase 2: Build gameweek history from the live API

The bootstrap endpoint gives current player totals, but not one row per player per gameweek. The next cell calls `element-summary/{id}/` for every player, flattens each response's `history` list, and saves the resulting modeling dataset.

In [ ]:
from src.data_cleaning import fetch_gameweek_history

history = fetch_gameweek_history(players=players)
history_path = PROJECT_ROOT / "data" / "processed" / "fpl_gameweek_history.csv"
history_path.parent.mkdir(parents=True, exist_ok=True)
history.to_csv(history_path, index=False)

print(f"Saved {len(history):,} rows for {history['player_id'].nunique():,} players to {history_path}")
history[[
    "player_id",
    "gameweek",
    "total_points",
    "minutes",
    "opponent_team",
    "was_home",
]].head()

## Phase 2: Leakage-safe model features

Rolling features use only information available before the current gameweek. The first row for each player is intentionally missing because there is no prior gameweek to average.

In [ ]:
from src.feature_engineering import add_minutes_reliability, add_rolling_form

history_features = add_rolling_form(history, window=3)
history_features = add_minutes_reliability(history_features, window=5)

history_features[[
    "player_id",
    "gameweek",
    "total_points",
    "rolling_form_3gw",
    "minutes_reliability",
]].head(10)

## Phase 2: Time-based baseline and linear regression

The split is chronological: earlier gameweeks train the model and the final available gameweek tests it. No rows are shuffled, so future information cannot enter training.

In [ ]:
from src.model import (
    evaluate_model,
    time_based_split,
    train_baseline_model,
    train_linear_regression,
)

latest_gameweek = int(history_features["gameweek"].max())
train_df, test_df = time_based_split(
    history_features,
    train_gw_end=latest_gameweek - 1,
    test_gw_start=latest_gameweek,
)
baseline_predictions = train_baseline_model(train_df)
features = ["rolling_form_3gw", "minutes_reliability"]
model = train_linear_regression(train_df, features, target="total_points")
metrics = evaluate_model(
    model,
    test_df,
    features,
    target="total_points",
    baseline_predictions=baseline_predictions,
)
pd.Series(metrics).round(3)